# v08 · Words outside the charset are filed by spelling, not by address

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
l3 = load("PtolemyDesktop/Callimachus/hyperwebster_layer3.py", "l3")
import tempfile, pathlib
root = pathlib.Path(tempfile.mkdtemp())
B = l3.HyperWebsterLayer3(words_root=str(root))
print("address_of('naïve'):", B.address_of("naïve"))       # None: 'ï' is not in the 97-symbol set
print("path:", B._record_path("naïve").name, "| in-charset path:", B._record_path("naive").name[:20] + "...")
# a word with symbols that are neither in the charset nor word characters is squashed to underscores
B.ingest([("a€b", 1.0, "first"), ("a£b", 2.0, "second")])
print(B._record_path("a€b").name, "|", B._record_path("a£b").name)
r = B.lookup("a€b")
print("record for 'a€b': word =", repr(r.word), "| hit_count =", r.hit_count)
assert B._record_path("a€b") == B._record_path("a£b") and r.hit_count == 2

address_of('naïve'): None
path: naïve.json | in-charset path: 00000000000000000000...
a_b.json | a_b.json
record for 'a€b': word = 'a€b' | hit_count = 2


Inside the charset the filename is the address, so two words never share a file. Outside it the fallback is `re.sub(r'[^\w]', '_', word)`: letters such as `ï` survive (Python's `\w` is Unicode-aware), symbols such as `€` and `£` both become `_`, so `a€b` and `a£b` land in one file and the second overwrites nothing but increments the first's hit count. The exact bijection covers only the 97-symbol charset. v09 (Unicode public charset in `charset.py`) is the later answer.